# Script 3 — Treinamento dos Modelos de ML (V5)
**TCC: Predição de Indicadores Financeiros Corporativos com ML e IA Generativa**

| Decisão | Justificativa |
|---------|---------------|
| **9 targets** | 3 primários (DRE) + 5 balanço + 1 caixa → habilita Z''-Score prospectivo completo |
| **Split temporal** | Treino ≤2022, Teste 2023–2024 — o modelo nunca viu dados futuros |
| **GroupKFold por empresa** | Evita vazamento temporal cruzado entre empresas no CV |
| **Transformação seletiva por target** | log1p para séries positivas e arcsinh para séries negativas/mistas |
| **SMAPE como métrica principal** | Definido para negativos (Lucro Líquido pode ser negativo) |
| **Theil's U** | Prova que ML supera baseline ingênua |
| **Acurácia Direcional** | Percentual de acertos na direção (sobe/desce) |
| **Imputer no Pipeline** | Evita leakage de NaN das features YoY no CV |
| **Curvas de aprendizado** | Diagnóstico automático de overfitting/underfitting |
| **Análise de resíduos** | Valida pressupostos e detecta padrões sistemáticos |

## Etapa 0 — Dependências e configuração

In [2]:
import logging, json, pickle, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, learning_curve
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, make_scorer
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:,.4f}'.format)

PASTA_SAIDA = Path('outputs')
PASTA_SAIDA.mkdir(exist_ok=True)
logger = logging.getLogger('pipeline_modelagem')
logger.setLevel(logging.DEBUG)
logger.handlers.clear()
_fmt = logging.Formatter('%(asctime)s | %(levelname)-8s | %(message)s',
                         datefmt='%Y-%m-%d %H:%M:%S')
_sh = logging.StreamHandler()
_sh.setLevel(logging.INFO)
_sh.setFormatter(_fmt)
logger.addHandler(_sh)
PASTA_LOGS = PASTA_SAIDA / 'logs'
PASTA_LOGS.mkdir(exist_ok=True)
_fh = logging.FileHandler(PASTA_LOGS / 'pipeline_modelagem.log',
                          mode='w', encoding='utf-8')
_fh.setLevel(logging.DEBUG)
_fh.setFormatter(_fmt)
logger.addHandler(_fh)
# ── Parâmetros ─────────────────────────────────────────────────────────────
N_SPLITS_WF  = 5        # número de folds do Walk-Forward CV
RANDOM_STATE = 42
ANO_CORTE    = 2022     # treino ≤ 2022, teste ≥ 2023
ANOS_COVID   = {2020, 2021}   # CORREÇÃO 7: anos atípicos — flag explícita
# Targets que recebem log1p (séries positivas e assimétricas)
LOG_TARGETS = {
    'TARGET_DRE_3.01', 'TARGET_EBITDA',
    'TARGET_BPA_1', 'TARGET_BPA_1.01',
    'TARGET_BPP_2.01', 'TARGET_BPP_2.03', 'TARGET_BPP_2',
}
# Targets com valores negativos ou mistos recebem arcsinh
ARCSINH_TARGETS = {
    'TARGET_DFC_MI_6.01',
    'TARGET_DRE_3.11',  # Lucro Líquido pode ser negativo
}
TRANSFORM_TARGETS = LOG_TARGETS | ARCSINH_TARGETS

def get_target_transform(target):
    if target in LOG_TARGETS:    return 'log1p'
    if target in ARCSINH_TARGETS: return 'arcsinh'
    return 'none'

def target_transform(y, transformacao='none'):
    y_arr = np.asarray(y, dtype=float)
    if not np.isfinite(y_arr).all():
        n_bad = np.size(y_arr) - np.isfinite(y_arr).sum()
        raise ValueError(f'target_transform: há {n_bad} valores não finitos.')
    if transformacao == 'log1p':
        if np.any(y_arr <= -1):
            raise ValueError(
                f"target_transform(log1p): valores <= -1 encontrados. "
                f"Use 'arcsinh' para targets com negativos/mistos."
            )
        return np.log1p(y_arr)
    if transformacao == 'arcsinh':
        return np.arcsinh(y_arr)
    return y_arr.copy()

def target_inverse_transform(y_pred, transformacao='none'):
    y_arr = np.asarray(y_pred, dtype=float)
    if transformacao == 'log1p':   return np.expm1(y_arr)
    if transformacao == 'arcsinh': return np.sinh(y_arr)
    return y_arr

logger.info("Script 3 iniciado | sklearn=%s", __import__('sklearn').__version__)
print("✅ Dependências carregadas")

2026-05-08 23:24:37 | INFO     | Script 3 iniciado | sklearn=1.8.0


✅ Dependências carregadas


## Etapa 1 — Carregamento e split temporal

In [3]:
dataset = pd.read_parquet(PASTA_SAIDA / 'dataset_preparado.parquet')
with open(PASTA_SAIDA / 'features.pkl',      'rb') as f: FEATURES      = pickle.load(f)
with open(PASTA_SAIDA / 'targets.pkl',       'rb') as f: TARGETS       = pickle.load(f)
with open(PASTA_SAIDA / 'kpis.pkl',          'rb') as f: KPIS          = pickle.load(f)
with open(PASTA_SAIDA / 'grupos_treino.pkl', 'rb') as f: GRUPOS_TREINO = pickle.load(f)
# ── Carregar splits do Script 2 ───────────────────────────────────────────
cam_treino = PASTA_SAIDA / 'treino.parquet'
cam_teste  = PASTA_SAIDA / 'teste.parquet'
if cam_treino.exists() and cam_teste.exists():
    treino = pd.read_parquet(cam_treino)
    teste  = pd.read_parquet(cam_teste)
    logger.info("Splits carregados dos parquets do Script 2")
else:
    logger.warning("treino.parquet não encontrado — recalculando split temporal")
    if 'split' not in dataset.columns:
        dataset['split'] = np.where(
            dataset['ANO'].astype(float) <= ANO_CORTE, 'treino',
            np.where(dataset['ANO'].astype(float) <= 2024, 'teste', 'prospectivo')
        )
    treino = dataset[dataset['split'] == 'treino'].reset_index(drop=True)
    teste  = dataset[dataset['split'] == 'teste'].reset_index(drop=True)
    GRUPOS_TREINO = treino['CNPJ_CIA'].values
    treino.to_parquet(cam_treino, index=False)
    teste.to_parquet(cam_teste,   index=False)
    with open(PASTA_SAIDA / 'grupos_treino.pkl', 'wb') as f:
        pickle.dump(GRUPOS_TREINO, f)
assert len(treino) > 0, "Treino vazio — verifique o Script 2"
assert len(teste)  > 0, "Teste vazio — verifique o Script 2"
# ── Verificação anti-leakage prospectivo ─────────────────────────────────
anos_treino = set(treino['ANO'].dropna().astype(int).unique())
anos_teste  = set(teste['ANO'].dropna().astype(int).unique())
anos_prosp  = {a for a in anos_treino | anos_teste if a >= 2025}
if anos_prosp:
    logger.error("Anos prospectivos vazaram para treino/teste: %s", anos_prosp)
else:
    logger.info("Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)")
# ── CORREÇÃO 3: diagnóstico de features temporais (lags/yoy/roll) ─────────
colunas_lag = [f for f in FEATURES if any(
    s in f for s in ['_lag', '_roll', '_diff1', '_growth1', '_yoy']
)]
logger.info("Features temporais em FEATURES: %d/%d", len(colunas_lag), len(FEATURES))
print(f"  Features temporais (lags/yoy/roll): {len(colunas_lag)} de {len(FEATURES)}")
if len(colunas_lag) == 0:
    logger.warning(
        "ATENÇÃO: nenhuma feature temporal encontrada em FEATURES. "
        "Verificar seleção de features no Script 2 (RFE pode ter excluído os lags)."
    )
# ── CORREÇÃO 7: flag_covid adicionada ao treino e teste se ausente ─────────
for df_name, df in [('treino', treino), ('teste', teste)]:
    if 'flag_covid' not in df.columns:
        df['flag_covid'] = df['ANO'].isin(ANOS_COVID).astype(float)
        logger.info("flag_covid adicionada ao %s", df_name)
if 'flag_covid' not in FEATURES:
    FEATURES = list(FEATURES) + ['flag_covid']
    logger.info("flag_covid adicionada à lista FEATURES")
logger.info("Treino: %d obs (≤%d) | Teste: %d obs (%d–2024)",
            len(treino), ANO_CORTE, len(teste), ANO_CORTE + 1)
print(f"\\n{'='*65}")
print(f"  Split temporal — carregado do Script 2")
print(f"{'='*65}")
print(f"  Treino (≤{ANO_CORTE}): {len(treino):>4} obs | "
      f"DFP={(treino['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(treino['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_treino)[:3]}...{sorted(anos_treino)[-1:]}")
print(f"  Teste ({ANO_CORTE+1}–2024): {len(teste):>4} obs | "
      f"DFP={(teste['ORIGEM']=='DFP').sum():>3} | "
      f"ITR={(teste['ORIGEM']=='ITR').sum():>3} | "
      f"anos {sorted(anos_teste)}")
print(f"  Prospectivo (≥2025): carregado pelo Script 5")
print(f"  Isolamento prospectivo: ✅")
anos_covid_no_treino = ANOS_COVID & anos_treino
print(f"  Anos COVID no treino: {sorted(anos_covid_no_treino)} → flag_covid=1")
print(f"{'='*65}")
print(f"\\nFeatures: {len(FEATURES)} | Targets: {len(TARGETS)}")
print(f"  → Lags/YoY/Roll: {len(colunas_lag)} features temporais")
print("Targets:")
for t in TARGETS:
    if t in treino.columns:
        if t in LOG_TARGETS:       transform_flag = "📐log   "
        elif t in ARCSINH_TARGETS: transform_flag = "📐arcsinh"
        else:                      transform_flag = "         "
        n_tr = treino[t].notna().sum()
        n_te = teste[t].notna().sum() if t in teste.columns else 0
        print(f"  {transform_flag} {t:<30} treino={n_tr:,} | teste={n_te:,}")

2026-05-08 23:24:55 | INFO     | Splits carregados dos parquets do Script 2
2026-05-08 23:24:55 | INFO     | Isolamento prospectivo: PASSOU ✅ (nenhum ano ≥2025 no treino/teste)
2026-05-08 23:24:55 | INFO     | Features temporais em FEATURES: 15/15
2026-05-08 23:24:55 | INFO     | flag_covid adicionada ao treino
2026-05-08 23:24:55 | INFO     | flag_covid adicionada ao teste
2026-05-08 23:24:55 | INFO     | flag_covid adicionada à lista FEATURES
2026-05-08 23:24:55 | INFO     | Treino: 713 obs (≤2022) | Teste: 168 obs (2023–2024)


  Features temporais (lags/yoy/roll): 15 de 15
\n=================================================================
  Split temporal — carregado do Script 2
  Treino (≤2022):  713 obs | DFP=181 | ITR=532 | anos [np.int64(2015), np.int64(2016), np.int64(2017)]...[np.int64(2022)]
  Teste (2023–2024):  168 obs | DFP= 24 | ITR=144 | anos [np.int64(2023), np.int64(2024)]
  Prospectivo (≥2025): carregado pelo Script 5
  Isolamento prospectivo: ✅
  Anos COVID no treino: [2020, 2021] → flag_covid=1
\nFeatures: 16 | Targets: 9
  → Lags/YoY/Roll: 15 features temporais
Targets:
  📐log    TARGET_DRE_3.01                treino=713 | teste=168
  📐arcsinh TARGET_DRE_3.11                treino=713 | teste=168
  📐log    TARGET_EBITDA                  treino=713 | teste=168
  📐log    TARGET_BPA_1                   treino=713 | teste=168
  📐log    TARGET_BPA_1.01                treino=713 | teste=168
  📐log    TARGET_BPP_2.01                treino=713 | teste=168
  📐log    TARGET_BPP_2.03                t

## Etapa 2 — Métricas e Baseline Ingênua

Além das métricas estatísticas clássicas, o pipeline calcula:

**Theil's U:** U<1 prova que o modelo supera a baseline de persistência.

**Acurácia Direcional:** % de acertos na direção (sobe/desce) — mais relevante para gestores.

**SMAPE:** erro percentual simétrico, definido mesmo para valores negativos (Lucro Líquido).

In [4]:
# %%
def smape(y_true, y_pred):
    """Symmetric MAPE — definido para valores negativos e próximos de zero."""
    num   = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else np.nan

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def theil_u(y_true, y_pred):
    n = len(y_true)
    if n < 2: return np.nan
    erro_modelo   = np.sqrt(np.mean((y_true[1:] - y_pred[1:])**2))
    erro_baseline = np.sqrt(np.mean((y_true[1:] - y_true[:-1])**2))
    return float(erro_modelo / erro_baseline) if erro_baseline > 0 else np.nan

def acuracia_direcional(y_true, y_pred):
    if len(y_true) < 2: return np.nan
    dir_real = np.sign(np.diff(y_true))
    dir_pred = np.sign(np.diff(y_pred))
    mask = dir_real != 0
    return float(np.mean(dir_real[mask] == dir_pred[mask])) if mask.sum() > 0 else np.nan

# ── CORREÇÃO 2: SMAPE como scorer para GridSearchCV ───────────────────────
# Garante que a otimização de hiperparâmetros trate grandes e pequenas empresas
# com o mesmo peso — eliminando o viés do RMSE absoluto.
def _smape_negativo(y_true, y_pred):
    """SMAPE negativo para make_scorer (sklearn maximiza, então negamos)."""
    num   = np.abs(y_true - y_pred)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    mask  = denom > 0
    return -float(np.mean(num[mask] / denom[mask])) if mask.sum() > 0 else 0.0
smape_scorer = make_scorer(_smape_negativo)

def calcular_baseline(treino_df, teste_df, target):
    """
    Baseline ingênua por empresa: persistência do último valor observado
    da própria companhia, respeitando a ordem temporal.
    """
    if target not in treino_df.columns or target not in teste_df.columns: return {}
    if 'CNPJ_CIA' not in treino_df.columns or 'CNPJ_CIA' not in teste_df.columns: return {}

    candidatos_tempo = [
        'DT_FIM_EXERC', 'DATA_REFERENCIA', 'DATA', 'DT_REFERENCIA',
        'TRIMESTRE', 'TRI', 'PERIODO', 'PERÍODO', 'ANO'
    ]
    time_col = next(
        (c for c in candidatos_tempo if c in treino_df.columns and c in teste_df.columns),
        None
    )
    cols_ordenacao = ['CNPJ_CIA']
    if time_col is not None:
        cols_ordenacao.append(time_col)

    treino_tmp = treino_df.reset_index(drop=True).copy()
    teste_tmp  = teste_df.reset_index(drop=True).copy()
    treino_tmp['_ordem_original'] = np.arange(len(treino_tmp))
    teste_tmp['_ordem_original']  = np.arange(len(teste_tmp))
    cols_select = list(dict.fromkeys(cols_ordenacao + ['_ordem_original', target]))

    base = pd.concat([
        treino_tmp[cols_select].assign(__split='treino'),
        teste_tmp[cols_select].assign(__split='teste'),
    ], ignore_index=True)
    base = base.sort_values(cols_ordenacao + ['_ordem_original'],
                            kind='mergesort').reset_index(drop=True)
    base['baseline_prev'] = (
        base.groupby('CNPJ_CIA')[target]
            .transform(lambda s: s.ffill().shift(1))
    )
    mask_teste  = base['__split'] == 'teste'
    mask_valido = mask_teste & base[target].notna() & base['baseline_prev'].notna()
    if mask_valido.sum() == 0: return {}

    y_t = base.loc[mask_valido, target].values
    y_p = base.loc[mask_valido, 'baseline_prev'].values
    cobertura = float(mask_valido.sum() / max(1, int(mask_teste.sum())))

    return {
        'RMSE_baseline'     : rmse(y_t, y_p),
        'MAE_baseline'      : float(mean_absolute_error(y_t, y_p)),
        'SMAPE_baseline'    : smape(y_t, y_p),
        'R2_baseline'       : float(r2_score(y_t, y_p)),
        'TheilU_baseline'   : 1.0,
        'DA_baseline'       : float(acuracia_direcional(y_t, y_p)),
        'Cobertura_baseline': cobertura,
        'TimeCol_baseline'  : time_col if time_col is not None else '',
    }

baselines = {}
print("=== Baseline Ingênua por empresa (persistência) ===")
print(f"  {'Target':<30} {'RMSE':>14} {'SMAPE':>7} {'R²':>6} "
      f"{'TheilU':>7} {'DA':>6} {'Cob.':>6} {'ColTempo'}")
print(f"  {'-'*30} {'-'*14} {'-'*7} {'-'*6} {'-'*7} {'-'*6} {'-'*6} {'-'*14}")
for t in TARGETS:
    b = calcular_baseline(treino, teste, t)
    baselines[t] = b
    if b:
        print(f"  {t:<30} {b['RMSE_baseline']:>14,.0f} "
              f"{b['SMAPE_baseline']:>7.1%} {b['R2_baseline']:>6.3f} "
              f"{b['TheilU_baseline']:>7.2f} {b['DA_baseline']:>6.1%} "
              f"{b.get('Cobertura_baseline', np.nan):>6.1%} "
              f"  {b.get('TimeCol_baseline', 'N/A')}")
    else:
        print(f"  {t:<30} N/A (sem dados suficientes)")

=== Baseline Ingênua por empresa (persistência) ===
  Target                                   RMSE   SMAPE     R²  TheilU     DA   Cob. ColTempo
  ------------------------------ -------------- ------- ------ ------- ------ ------ --------------
  TARGET_DRE_3.01                    22,487,246   17.6%  0.959    1.00  59.7%  85.7%   TRIMESTRE
  TARGET_DRE_3.11                    21,816,677   71.2% -0.689    1.00  74.8%  85.7%   TRIMESTRE
  TARGET_EBITDA                       6,991,551   21.5%  0.807    1.00  64.3%  85.7%   TRIMESTRE
  TARGET_BPA_1                       23,200,605   17.7%  0.990    1.00  89.1%  85.7%   TRIMESTRE
  TARGET_BPA_1.01                     5,357,315   18.7%  0.971    1.00  61.3%  85.7%   TRIMESTRE
  TARGET_BPP_2.01                     6,304,194   25.5%  0.973    1.00  66.4%  85.7%   TRIMESTRE
  TARGET_BPP_2.03                     6,531,550   22.1%  0.993    1.00  74.8%  85.7%   TRIMESTRE
  TARGET_BPP_2                       23,200,605   17.7%  0.990    1.00  89.

## Etapa 3 — Walk-Forward Cross-Validation (substitui GroupKFold)

**CORREÇÃO 1:** O GroupKFold original era aleatório por empresa e não respeitava
a ordem temporal dentro do período de treino — dados de 2022 podiam cair no
fold de validação enquanto dados de 2019 ficavam no treino (data leakage temporal).

O Walk-Forward CV usa janela expansível por ano:
  Fold 1: treino anos[0..k-1]  | valida anos[k]
  Fold 2: treino anos[0..k]    | valida anos[k+1]
  ...
Isso garante que o modelo NUNCA valida em dados do passado em relação ao treino.

In [5]:
def criar_folds_walkforward(df, col_ano='ANO', n_splits=N_SPLITS_WF):
    """
    Cria folds de Walk-Forward (janela expansível) baseados no ANO.
    Retorna lista de (idx_treino, idx_validacao) respeitando ordem temporal.
    """
    anos = sorted(df[col_ano].dropna().astype(int).unique())

    if len(anos) < n_splits + 1:
        n_splits = max(1, len(anos) - 1)
        logger.warning("Walk-Forward: anos insuficientes, reduzindo para %d folds", n_splits)

    anos_val = anos[-n_splits:]
    folds = []
    for ano_val in anos_val:
        idx_tr  = np.where(df[col_ano].astype(float) < ano_val)[0]
        idx_val = np.where(df[col_ano].astype(float) == ano_val)[0]
        if len(idx_tr) > 0 and len(idx_val) > 0:
            folds.append((idx_tr, idx_val))
            logger.debug("Fold WF: treino anos<%d (%d obs) | val %d (%d obs)",
                         ano_val, len(idx_tr), ano_val, len(idx_val))

    logger.info("Walk-Forward CV: %d folds | anos validação: %s", len(folds), anos_val)
    return folds

# ── Ridge ──────────────────────────────────────────────────────────────────
est_ridge = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("ridge",   Ridge()),
])
grade_ridge = {"ridge__alpha": [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]}
# ── SVR ────────────────────────────────────────────────────────────────────
est_svr = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler",  StandardScaler()),
    ("svr",     SVR(kernel="rbf", max_iter=20000)),
])
grade_svr = {
    "svr__C":       [0.1, 1.0, 10.0, 100.0],
    "svr__epsilon": [0.01, 0.05, 0.1, 0.5],
    "svr__gamma":   ["scale", "auto"],
}
# ── Random Forest ──────────────────────────────────────────────────────────
est_rf = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("rf",      RandomForestRegressor(n_estimators=300,
                                      random_state=RANDOM_STATE, n_jobs=-1)),
])
grade_rf = {
    "rf__max_depth":        [None, 5, 10, 20],
    "rf__min_samples_leaf": [1, 2, 5],
    "rf__max_features":     ["sqrt", "log2"],
}
# ── Gradient Boosting ──────────────────────────────────────────────────────
est_gb = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("gb",      GradientBoostingRegressor(random_state=RANDOM_STATE)),
])
grade_gb = {
    "gb__n_estimators":  [100, 200, 300],
    "gb__learning_rate": [0.01, 0.05, 0.1, 0.2],
    "gb__max_depth":     [3, 5],
    "gb__subsample":     [0.7, 0.8, 1.0],
}
ALGORITMOS = {
    "Ridge":            (est_ridge, grade_ridge),
    "SVR":              (est_svr,   grade_svr),
    "RandomForest":     (est_rf,    grade_rf),
    "GradientBoosting": (est_gb,    grade_gb),
}
logger.info("%d algoritmos | Walk-Forward CV (n_splits=%d)", len(ALGORITMOS), N_SPLITS_WF)
print(f"✅ {len(ALGORITMOS)} algoritmos com Walk-Forward CV (n_splits={N_SPLITS_WF})")
print(f"✅ Scoring: SMAPE (proporcional ao tamanho da empresa)")

2026-05-08 23:25:12 | INFO     | 4 algoritmos | Walk-Forward CV (n_splits=5)


✅ 4 algoritmos com Walk-Forward CV (n_splits=5)
✅ Scoring: SMAPE (proporcional ao tamanho da empresa)


## Etapa 4 — Treinamento com Walk-Forward Nested Cross-Validation

 Estrutura de dois loops aninhados:

   Loop externo (Walk-Forward): estima generalização temporal
     Para cada fold (ano_k como validação, todos os anos < ano_k como treino):

       Loop interno (GridSearchCV com Walk-Forward interno):
         Seleciona hiperparâmetros usando SMAPE como scoring
         Folds internos criados sobre o subconjunto externo de treino

       Avalia o modelo com melhores hiperparâmetros no fold externo
       Métricas calculadas no espaço original (transformação revertida)

   Fit final no conjunto completo de treino (GridSearchCV com WF)

In [6]:
def treinar_alg(nome, estimador, grade, df_treino_completo,
                target, features, transformacao='none', n_splits_wf=N_SPLITS_WF):
    """
    Treinamento com Walk-Forward Nested CV.

    Loop externo : Walk-Forward por ano — respeita ordem temporal.
    Loop interno : GridSearchCV com Walk-Forward interno — scoring SMAPE.
    """
    df_t = df_treino_completo[features + [target, 'ANO']].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)

    X_full     = df_t[features].values
    y_full     = df_t[target].values
    y_fit_full = target_transform(y_full, transformacao)

    folds_ext = criar_folds_walkforward(df_t, col_ano='ANO', n_splits=n_splits_wf)

    if len(folds_ext) == 0:
        logger.warning("%s | %s: sem folds válidos no Walk-Forward — fallback cv=3", nome, target)
        gs_fb = GridSearchCV(estimador, grade, cv=3, scoring=smape_scorer,
                             refit=True, n_jobs=-1, verbose=0)
        gs_fb.fit(X_full, y_fit_full)
        melhor = gs_fb.best_estimator_
        nan_m = {k: np.nan for k in ['RMSE_CV','RMSE_CV_std','MAE_CV','SMAPE_CV',
                                      'SMAPE_CV_std','R2_CV','R2_CV_std','TheilU_CV','DA_CV']}
        nan_m.update({'transformacao': transformacao, 'log_transform': transformacao=='log1p',
                      'best_params': gs_fb.best_params_, 'n_folds_wf': 0})
        return melhor, nan_m

    rmse_v, mae_v, smape_v, r2_v, theil_v, da_v = [], [], [], [], [], []

    for tr_idx_ext, val_idx_ext in folds_ext:
        X_tr_ext   = X_full[tr_idx_ext]
        y_tr_ext   = y_fit_full[tr_idx_ext]
        y_orig_val = y_full[val_idx_ext]

        # Loop interno: Walk-Forward sobre o subconjunto externo de treino
        df_sub    = df_t.iloc[tr_idx_ext].reset_index(drop=True)
        folds_int = criar_folds_walkforward(df_sub, col_ano='ANO',
                                            n_splits=max(2, n_splits_wf - 1))
        cv_int = folds_int if len(folds_int) >= 2 else 3

        gs = GridSearchCV(estimador, grade, cv=cv_int,
                          scoring=smape_scorer,   # CORREÇÃO 2
                          refit=True, n_jobs=-1, verbose=0)
        gs.fit(X_tr_ext, y_tr_ext)
        melhor_fold = gs.best_estimator_

        y_pred_raw = melhor_fold.predict(X_full[val_idx_ext])
        y_pred     = target_inverse_transform(y_pred_raw, transformacao)

        rmse_v.append(rmse(y_orig_val, y_pred))
        mae_v.append(float(mean_absolute_error(y_orig_val, y_pred)))
        smape_v.append(smape(y_orig_val, y_pred))
        r2_v.append(float(r2_score(y_orig_val, y_pred)))
        theil_v.append(theil_u(y_orig_val, y_pred))
        da_v.append(acuracia_direcional(y_orig_val, y_pred))

    # Fit final: GridSearchCV com WF sobre todos os dados de treino
    folds_final = criar_folds_walkforward(df_t, col_ano='ANO',
                                          n_splits=max(2, n_splits_wf - 1))
    cv_final = folds_final if len(folds_final) >= 2 else 3
    gs_final = GridSearchCV(estimador, grade, cv=cv_final, scoring=smape_scorer,
                            refit=True, n_jobs=-1, verbose=0)
    gs_final.fit(X_full, y_fit_full)
    melhor = gs_final.best_estimator_

    def _m(lst): return float(np.nanmean(lst))
    def _s(lst): return float(np.nanstd(lst))

    metricas = {
        'RMSE_CV'      : _m(rmse_v),   'RMSE_CV_std'  : _s(rmse_v),
        'MAE_CV'       : _m(mae_v),
        'SMAPE_CV'     : _m(smape_v),  'SMAPE_CV_std' : _s(smape_v),
        'R2_CV'        : _m(r2_v),     'R2_CV_std'    : _s(r2_v),
        'TheilU_CV'    : _m(theil_v),
        'DA_CV'        : _m(da_v),
        'transformacao': transformacao,
        'log_transform': transformacao == 'log1p',
        'best_params'  : gs_final.best_params_,
        'n_folds_wf'   : len(rmse_v),
    }

    flag_theil = "✅" if metricas['TheilU_CV'] < 1 else "⚠️"
    logger.info("  %-20s RMSE=%10.0f±%8.0f  SMAPE=%5.1f%%  "
                "R²=%5.3f  TheilU=%s%.3f  DA=%.1f%%  transf=%s  folds=%d",
                nome, metricas['RMSE_CV'], metricas['RMSE_CV_std'],
                metricas['SMAPE_CV']*100, metricas['R2_CV'],
                flag_theil, metricas['TheilU_CV'],
                metricas['DA_CV']*100, transformacao, metricas['n_folds_wf'])
    print(f"  {flag_theil} {nome:<20} "
          f"RMSE={metricas['RMSE_CV']:>12,.0f}  "
          f"SMAPE={metricas['SMAPE_CV']:>5.1%}  "
          f"R²={metricas['R2_CV']:>6.3f}  "
          f"U={metricas['TheilU_CV']:.3f}  "
          f"DA={metricas['DA_CV']:.1%}  "
          f"folds={metricas['n_folds_wf']}")

    return melhor, metricas

# ── Execução ──────────────────────────────────────────────────────────────
resultados = {}
for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})

    print(f"\\n{'='*72}")
    print(f"  TARGET: {target}  |  transform={transformacao}")
    if b:
        print(f"  Baseline → RMSE={b.get('RMSE_baseline', 0):,.0f}  "
              f"SMAPE={b.get('SMAPE_baseline', 0):.1%}  "
              f"R²={b.get('R2_baseline', 0):.3f}  "
              f"DA={b.get('DA_baseline', 0):.1%}  "
              f"Cob.={b.get('Cobertura_baseline', np.nan):.1%}")
    print(f"  {'Alg':<22} {'RMSE':>14} {'SMAPE':>7} {'R²':>7} "
          f"{'TheilU':>7} {'DA':>6} {'Folds':>6}")
    print(f"  {'-'*22} {'-'*14} {'-'*7} {'-'*7} {'-'*7} {'-'*6} {'-'*6}")

    resultados[target] = {}
    for nome, (est, grade) in ALGORITMOS.items():
        modelo, metricas = treinar_alg(
            nome, est, grade,
            df_treino_completo=treino,
            target=target,
            features=FEATURES,
            transformacao=transformacao,
            n_splits_wf=N_SPLITS_WF,
        )
        resultados[target][nome] = (modelo, metricas)
        joblib.dump(
            {'modelo': modelo, 'transformacao': transformacao,
             'log_transform': transformacao == 'log1p', 'features': FEATURES},
            PASTA_SAIDA / f'modelo_{target}_{nome}.pkl'
        )

    logger.info("TARGET %s concluído", target)
print("\\n✅ Treinamento concluído para todos os targets.")

2026-05-08 23:25:18 | INFO     | Walk-Forward CV: 5 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:25:18 | WARNING  | Walk-Forward: anos insuficientes, reduzindo para 2 folds
2026-05-08 23:25:18 | INFO     | Walk-Forward CV: 2 folds | anos validação: [np.int64(2016), np.int64(2017)]


\n========================================================================
  TARGET: TARGET_DRE_3.01  |  transform=log1p
  Baseline → RMSE=22,487,246  SMAPE=17.6%  R²=0.959  DA=59.7%  Cob.=85.7%
  Alg                              RMSE   SMAPE      R²  TheilU     DA  Folds
  ---------------------- -------------- ------- ------- ------- ------ ------


2026-05-08 23:25:23 | WARNING  | Walk-Forward: anos insuficientes, reduzindo para 3 folds
2026-05-08 23:25:23 | INFO     | Walk-Forward CV: 3 folds | anos validação: [np.int64(2016), np.int64(2017), np.int64(2018)]
2026-05-08 23:25:23 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
2026-05-08 23:25:23 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
2026-05-08 23:25:24 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
2026-05-08 23:25:24 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:25:24 | INFO     |   Ridge                RMSE=2974362197±3169920136  SMAPE= 59.6%  R²=-2846.312  TheilU=⚠️43.361  DA=87.0%  transf=log1p  folds=5
2026-05-08 23:25:24 | INFO     | Walk-Forward CV: 5 folds | 

  ⚠️ Ridge                RMSE=2,974,362,197  SMAPE=59.6%  R²=-2846.312  U=43.361  DA=87.0%  folds=5


2026-05-08 23:25:24 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
2026-05-08 23:25:24 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
2026-05-08 23:25:25 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
2026-05-08 23:25:25 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:25:26 | INFO     |   SVR                  RMSE=  72626170±24109475  SMAPE= 30.4%  R²=0.403  TheilU=⚠️1.089  DA=91.0%  transf=log1p  folds=5
2026-05-08 23:25:26 | INFO     | Walk-Forward CV: 5 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:25:26 | WARNING  | Walk-Forward: anos insuficientes, reduzindo para 2 folds
2026-05-08 23:25:26 | INFO     | Wa

...

  ⚠️ SVR                  RMSE=  32,278,758  SMAPE=65.4%  R²= 0.190  U=1.162  DA=64.3%  folds=5


2026-05-08 23:46:18 | WARNING  | Walk-Forward: anos insuficientes, reduzindo para 3 folds
2026-05-08 23:46:18 | INFO     | Walk-Forward CV: 3 folds | anos validação: [np.int64(2016), np.int64(2017), np.int64(2018)]
2026-05-08 23:46:27 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
2026-05-08 23:46:40 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
2026-05-08 23:46:55 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
2026-05-08 23:47:10 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:47:26 | INFO     |   RandomForest         RMSE=  10961703± 4983093  SMAPE= 68.8%  R²=0.904  TheilU=✅0.377  DA=59.7%  transf=arcsinh  folds=5
2026-05-08 23:47:26 | INFO     | Walk-Forward CV: 5 folds | anos v

  ✅ RandomForest         RMSE=  10,961,703  SMAPE=68.8%  R²= 0.904  U=0.377  DA=59.7%  folds=5


2026-05-08 23:47:31 | WARNING  | Walk-Forward: anos insuficientes, reduzindo para 3 folds
2026-05-08 23:47:31 | INFO     | Walk-Forward CV: 3 folds | anos validação: [np.int64(2016), np.int64(2017), np.int64(2018)]
2026-05-08 23:47:40 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019)]
2026-05-08 23:47:53 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
2026-05-08 23:48:09 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021)]
2026-05-08 23:48:28 | INFO     | Walk-Forward CV: 4 folds | anos validação: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:48:51 | INFO     |   GradientBoosting     RMSE=  16348740± 5450920  SMAPE= 50.4%  R²=0.710  TheilU=✅0.640  DA=62.1%  transf=arcsinh  folds=5
2026-05-08 23:48:51 | INFO     | TARGET TARGET_DFC_MI_6.01 concluí

  ✅ GradientBoosting     RMSE=  16,348,740  SMAPE=50.4%  R²= 0.710  U=0.640  DA=62.1%  folds=5
\n✅ Treinamento concluído para todos os targets.


## Etapa 5 — Avaliação no conjunto de teste hold-out

In [7]:
def avaliar_teste(modelo, X_te, y_te, transformacao):
    y_pred_raw = modelo.predict(X_te)
    y_pred     = target_inverse_transform(y_pred_raw, transformacao)
    mask = np.isfinite(y_te) & np.isfinite(y_pred)
    yt, yp = y_te[mask], y_pred[mask]
    return {
        'RMSE_teste'  : rmse(yt, yp),
        'MAE_teste'   : float(mean_absolute_error(yt, yp)),
        'SMAPE_teste' : smape(yt, yp),
        'R2_teste'    : float(r2_score(yt, yp)),
        'TheilU_teste': theil_u(yt, yp),
        'DA_teste'    : acuracia_direcional(yt, yp),
    }

print("\\n=== Avaliação no Teste Hold-out (2023–2024) ===")
metricas_teste = {}
for target in TARGETS:
    transformacao = get_target_transform(target)
    b = baselines.get(target, {})
    df_te = teste[FEATURES + [target]].copy()
    df_te = df_te[df_te[target].notna()]
    X_te  = df_te[FEATURES].values
    y_te  = df_te[target].values

    metricas_teste[target] = {}
    baseline_rmse = b.get('RMSE_baseline', np.inf)

    print(f"\\n{target}  (baseline RMSE={baseline_rmse:,.0f}  "
          f"DA={b.get('DA_baseline', 0):.1%}  "
          f"Cob.={b.get('Cobertura_baseline', np.nan):.1%})")
    print(f"  {'Algoritmo':<20} {'RMSE':>14} {'SMAPE':>7} "
          f"{'R²':>7} {'TheilU':>7} {'DA':>6} {'Bateu?':>7}")
    print(f"  {'-'*20} {'-'*14} {'-'*7} {'-'*7} {'-'*7} {'-'*6} {'-'*7}")

    for nome, (modelo, _) in resultados[target].items():
        m = avaliar_teste(modelo, X_te, y_te, transformacao)
        metricas_teste[target][nome] = m
        bateu    = m['RMSE_teste'] < baseline_rmse
        theil_ok = (m['TheilU_teste'] or 1.0) < 1.0
        flag = "✅" if bateu and theil_ok else ("🟡" if bateu else "❌")
        print(f"  {flag} {nome:<18} {m['RMSE_teste']:>14,.0f} "
              f"{m['SMAPE_teste']:>7.1%} {m['R2_teste']:>7.3f} "
              f"{m['TheilU_teste']:>7.3f} {m['DA_teste']:>6.1%} "
              f"{'✅' if bateu else '❌':>7}")
        logger.info("Teste | %s | %s: RMSE=%.0f SMAPE=%.2f%% R2=%.3f TheilU=%.3f DA=%.1f%%",
                    target, nome, m['RMSE_teste'], m['SMAPE_teste']*100,
                    m['R2_teste'], m['TheilU_teste'], m['DA_teste']*100)

2026-05-08 23:49:15 | INFO     | Teste | TARGET_DRE_3.01 | Ridge: RMSE=689657990 SMAPE=55.37% R2=-37.111 TheilU=12.540 DA=91.5%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_DRE_3.01 | SVR: RMSE=59233482 SMAPE=21.34% R2=0.719 TheilU=1.077 DA=93.6%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_DRE_3.01 | RandomForest: RMSE=14840096 SMAPE=6.52% R2=0.982 TheilU=0.270 DA=87.2%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_DRE_3.01 | GradientBoosting: RMSE=16533548 SMAPE=4.15% R2=0.978 TheilU=0.301 DA=100.0%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_DRE_3.11 | Ridge: RMSE=505917207 SMAPE=182.04% R2=-761.209 TheilU=39.302 DA=74.5%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_DRE_3.11 | SVR: RMSE=12663296 SMAPE=64.32% R2=0.522 TheilU=0.984 DA=72.3%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_DRE_3.11 | RandomForest: RMSE=8344892 SMAPE=64.27% R2=0.793 TheilU=0.648 DA=66.0%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_DRE_3.11 | GradientBoosting: RMSE=6231166 SMAPE=39.31% 

\n=== Avaliação no Teste Hold-out (2023–2024) ===
\nTARGET_DRE_3.01  (baseline RMSE=22,487,246  DA=59.7%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 689,657,990   55.4% -37.111  12.540  91.5%       ❌
  ❌ SVR                    59,233,482   21.3%   0.719   1.077  93.6%       ❌
  ✅ RandomForest           14,840,096    6.5%   0.982   0.270  87.2%       ✅
  ✅ GradientBoosting       16,533,548    4.1%   0.978   0.301 100.0%       ✅
\nTARGET_DRE_3.11  (baseline RMSE=21,816,677  DA=74.8%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 505,917,207  182.0% -761.209  39.302  74.5%       ❌
  ✅ SVR                    12,663,296   64.3%   0.522   0.984  72.3%       ✅
  ✅ RandomForest            8,344,892   64.

2026-05-08 23:49:15 | INFO     | Teste | TARGET_EBITDA | RandomForest: RMSE=5896984 SMAPE=21.06% R2=0.863 TheilU=0.868 DA=88.1%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_EBITDA | GradientBoosting: RMSE=5653506 SMAPE=20.52% R2=0.874 TheilU=0.832 DA=83.3%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPA_1 | Ridge: RMSE=146082750 SMAPE=62.52% R2=0.581 TheilU=1.200 DA=70.2%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPA_1 | SVR: RMSE=144205195 SMAPE=28.65% R2=0.592 TheilU=1.184 DA=74.5%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPA_1 | RandomForest: RMSE=28423928 SMAPE=9.82% R2=0.984 TheilU=0.233 DA=87.2%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPA_1 | GradientBoosting: RMSE=33718293 SMAPE=7.06% R2=0.978 TheilU=0.277 DA=91.5%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPA_1.01 | Ridge: RMSE=37758919 SMAPE=48.01% R2=-0.401 TheilU=2.359 DA=72.3%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPA_1.01 | SVR: RMSE=14516829 SMAPE=19.62% R2=0.793 TheilU=0.907 DA

  ✅ RandomForest            5,896,984   21.1%   0.863   0.868  88.1%       ✅
  ✅ GradientBoosting        5,653,506   20.5%   0.874   0.832  83.3%       ✅
\nTARGET_BPA_1  (baseline RMSE=23,200,605  DA=89.1%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 146,082,750   62.5%   0.581   1.200  70.2%       ❌
  ❌ SVR                   144,205,195   28.7%   0.592   1.184  74.5%       ❌
  ❌ RandomForest           28,423,928    9.8%   0.984   0.233  87.2%       ❌
  ❌ GradientBoosting       33,718,293    7.1%   0.978   0.277  91.5%       ❌
\nTARGET_BPA_1.01  (baseline RMSE=5,357,315  DA=61.3%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  37,758,919   48.0%  -0.401   2.359  72.3%       ❌
  ❌ SVR              

2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPP_2.01 | RandomForest: RMSE=7472025 SMAPE=9.97% R2=0.961 TheilU=0.381 DA=74.5%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPP_2.01 | GradientBoosting: RMSE=7133098 SMAPE=8.06% R2=0.964 TheilU=0.363 DA=83.0%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPP_2.03 | Ridge: RMSE=49546453 SMAPE=59.94% R2=0.622 TheilU=1.125 DA=83.0%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPP_2.03 | SVR: RMSE=41516629 SMAPE=28.76% R2=0.734 TheilU=0.943 DA=78.7%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPP_2.03 | RandomForest: RMSE=10159347 SMAPE=21.04% R2=0.984 TheilU=0.231 DA=83.0%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPP_2.03 | GradientBoosting: RMSE=11109407 SMAPE=17.79% R2=0.981 TheilU=0.252 DA=83.0%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPP_2 | Ridge: RMSE=146082750 SMAPE=62.52% R2=0.581 TheilU=1.200 DA=70.2%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_BPP_2 | SVR: RMSE=144205195 SMAPE=28.65% R2=0.592 TheilU

  ❌ RandomForest            7,472,025   10.0%   0.961   0.381  74.5%       ❌
  ❌ GradientBoosting        7,133,098    8.1%   0.964   0.363  83.0%       ❌
\nTARGET_BPP_2.03  (baseline RMSE=6,531,550  DA=74.8%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                  49,546,453   59.9%   0.622   1.125  83.0%       ❌
  ❌ SVR                    41,516,629   28.8%   0.734   0.943  78.7%       ❌
  ❌ RandomForest           10,159,347   21.0%   0.984   0.231  83.0%       ❌
  ❌ GradientBoosting       11,109,407   17.8%   0.981   0.252  83.0%       ❌
\nTARGET_BPP_2  (baseline RMSE=23,200,605  DA=89.1%  Cob.=85.7%)
  Algoritmo                      RMSE   SMAPE      R²  TheilU     DA  Bateu?
  -------------------- -------------- ------- ------- ------- ------ -------
  ❌ Ridge                 146,082,750   62.5%   0.581   1.200  70.2%       ❌
  ❌ SVR              

2026-05-08 23:49:15 | INFO     | Teste | TARGET_DFC_MI_6.01 | RandomForest: RMSE=6068114 SMAPE=62.67% R2=0.979 TheilU=0.259 DA=68.1%
2026-05-08 23:49:15 | INFO     | Teste | TARGET_DFC_MI_6.01 | GradientBoosting: RMSE=7853924 SMAPE=21.54% R2=0.965 TheilU=0.336 DA=68.1%


  ✅ RandomForest            6,068,114   62.7%   0.979   0.259  68.1%       ✅
  ✅ GradientBoosting        7,853,924   21.5%   0.965   0.336  68.1%       ✅


## Etapa 5B - Seleção do melhor modelo

CORREÇÃO 6: critério primário agora é SMAPE_CV (proporcional ao tamanho)
em vez de RMSE_CV (absoluto, favorece grandes empresas).
O conjunto de teste NÃO é consultado para esta decisão.

In [8]:
def escolher_melhor_modelo_cv(resultados_target):
    """
    Escolhe o melhor algoritmo usando apenas métricas de CV.
    Critério principal : menor SMAPE_CV  (proporcional ao tamanho da empresa).
    Desempate 1        : menor TheilU_CV.
    Desempate 2        : menor RMSE_CV.
    """
    return min(
        resultados_target.items(),
        key=lambda item: (
            item[1][1].get('SMAPE_CV',  np.inf),   # CORREÇÃO 6
            item[1][1].get('TheilU_CV', np.inf),
            item[1][1].get('RMSE_CV',   np.inf),
        )
    )[0]

melhores = {t: escolher_melhor_modelo_cv(resultados[t]) for t in TARGETS}
print("\\n=== Melhor modelo por target (critério: SMAPE_CV) ===")
for t, alg in melhores.items():
    m_cv   = resultados[t][alg][1]
    m_test = metricas_teste[t][alg]
    print(f"  {t:<35} → {alg:<20} "
          f"SMAPE_CV={m_cv['SMAPE_CV']:.1%}  "
          f"TheilU_CV={m_cv['TheilU_CV']:.3f}  "
          f"SMAPE_teste={m_test['SMAPE_teste']:.1%}")

\n=== Melhor modelo por target (critério: SMAPE_CV) ===
  TARGET_DRE_3.01                     → GradientBoosting     SMAPE_CV=8.6%  TheilU_CV=0.320  SMAPE_teste=4.1%
  TARGET_DRE_3.11                     → GradientBoosting     SMAPE_CV=50.2%  TheilU_CV=15.455  SMAPE_teste=39.3%
  TARGET_EBITDA                       → GradientBoosting     SMAPE_CV=25.0%  TheilU_CV=0.458  SMAPE_teste=20.5%
  TARGET_BPA_1                        → GradientBoosting     SMAPE_CV=10.8%  TheilU_CV=0.197  SMAPE_teste=7.1%
  TARGET_BPA_1.01                     → GradientBoosting     SMAPE_CV=11.8%  TheilU_CV=0.270  SMAPE_teste=6.0%
  TARGET_BPP_2.01                     → GradientBoosting     SMAPE_CV=14.2%  TheilU_CV=0.278  SMAPE_teste=8.1%
  TARGET_BPP_2.03                     → GradientBoosting     SMAPE_CV=17.9%  TheilU_CV=0.179  SMAPE_teste=17.8%
  TARGET_BPP_2                        → GradientBoosting     SMAPE_CV=10.8%  TheilU_CV=0.197  SMAPE_teste=7.1%
  TARGET_DFC_MI_6.01                  → GradientBoost

## Etapa 6 — Feature Importance

In [9]:
def extrair_importancia(modelo, features, nome_alg):
    step = [s for s, _ in modelo.steps][-1]
    est_final = modelo.named_steps[step]
    if hasattr(est_final, 'feature_importances_'):
        imp = est_final.feature_importances_
    elif hasattr(est_final, 'coef_'):
        imp = np.abs(est_final.coef_)
    else:
        return pd.Series(dtype=float)
    return pd.Series(imp, index=features).sort_values(ascending=False)

feature_importances = {}
print("\\n=== Feature Importance — Melhor Modelo por Target (SMAPE_CV) ===")
n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 1, figsize=(11, 5*n_t))
if n_t == 1: axes = [axes]
for i, target in enumerate(TARGETS):
    melhor_nome = melhores[target]
    melhor_mod  = resultados[target][melhor_nome][0]
    imp = extrair_importancia(melhor_mod, FEATURES, melhor_nome)
    feature_importances[target] = {'algoritmo': melhor_nome, 'importancias': imp.to_dict()}

    if not imp.empty:
        top = imp.head(min(12, len(imp)))
        colors = ['#1f4e79' if v == top.values[0] else
                  '#2e75b6' if v >= top.values[0]*0.7 else '#9dc3e6'
                  for v in top.values[::-1]]
        axes[i].barh(range(len(top)), top.values[::-1], color=colors, alpha=0.9)
        axes[i].set_yticks(range(len(top)))
        axes[i].set_yticklabels(top.index[::-1], fontsize=9)
        smape_val = metricas_teste[target][melhor_nome]['SMAPE_teste']
        axes[i].set_title(
            f"{target.replace('TARGET_', '')} — {melhor_nome} (SMAPE_teste={smape_val:.1%})",
            fontsize=11, fontweight='bold'
        )
        axes[i].set_xlabel('Importância Relativa')
        axes[i].grid(axis='x', alpha=0.3)
        for j, v in enumerate(top.values[::-1]):
            axes[i].text(v + imp.max()*0.005, j, f'{v:.3f}', va='center', fontsize=8)

    top3 = list(imp.head(3).index)
    top3_flags = ['[LAG]' if any(s in f for s in ['_lag','_roll','_yoy']) else '' for f in top3]
    print(f"  {target}: {melhor_nome} | top3={[f+g for f,g in zip(top3,top3_flags)]}")
plt.suptitle('Feature Importance — Melhor Modelo por Target (SMAPE_CV)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: feature_importance.png")

\n=== Feature Importance — Melhor Modelo por Target (SMAPE_CV) ===
  TARGET_DRE_3.01: GradientBoosting | top3=['TARGET_DRE_3.01_lag1[LAG]', 'TARGET_DRE_3.01_diff1', 'TARGET_BPA_1.01_lag1[LAG]']
  TARGET_DRE_3.11: GradientBoosting | top3=['TARGET_DRE_3.11_lag1[LAG]', 'TARGET_DRE_3.01_diff1', 'TARGET_BPA_1.01_lag2[LAG]']
  TARGET_EBITDA: GradientBoosting | top3=['TARGET_BPP_2_lag1[LAG]', 'TARGET_BPA_1_lag1[LAG]', 'TARGET_DFC_MI_6.01_lag1[LAG]']
  TARGET_BPA_1: GradientBoosting | top3=['TARGET_BPP_2_lag1[LAG]', 'TARGET_BPA_1_lag1[LAG]', 'TARGET_BPA_1.01_lag1[LAG]']
  TARGET_BPA_1.01: GradientBoosting | top3=['TARGET_BPA_1.01_lag1[LAG]', 'TARGET_BPP_2.01_lag1[LAG]', 'TARGET_BPA_1_lag1[LAG]']
  TARGET_BPP_2.01: GradientBoosting | top3=['TARGET_BPP_2.01_lag1[LAG]', 'TARGET_BPA_1.01_lag1[LAG]', 'TARGET_BPA_1_lag1[LAG]']
  TARGET_BPP_2.03: GradientBoosting | top3=['TARGET_BPP_2_lag1[LAG]', 'TARGET_BPA_1_lag1[LAG]', 'TARGET_BPP_2.03_lag2[LAG]']
  TARGET_BPP_2: GradientBoosting | top3=['TARGET_B

## Etapa 7 — Curvas de Aprendizado

In [10]:
print("\\nGerando curvas de aprendizado...")
n_t = len(TARGETS)
fig, axes = plt.subplots(1, n_t, figsize=(7*n_t, 5))
if n_t == 1: axes = [axes]
for i, target in enumerate(TARGETS):
    transformacao = get_target_transform(target)
    df_t = treino[FEATURES + [target, 'ANO']].copy()
    df_t = df_t[df_t[target].notna()].reset_index(drop=True)
    X = df_t[FEATURES].values
    y = target_transform(df_t[target].values, transformacao)

    melhor_nome = melhores[target]
    melhor_mod  = resultados[target][melhor_nome][0]

    folds_lc = criar_folds_walkforward(df_t, col_ano='ANO', n_splits=N_SPLITS_WF)
    cv_lc = folds_lc if len(folds_lc) >= 2 else 3

    try:
        sizes, tr_sc, val_sc = learning_curve(
            melhor_mod, X, y,
            cv=cv_lc,
            scoring='r2',
            train_sizes=np.linspace(0.3, 1.0, 5),
            n_jobs=-1,
        )
        ax = axes[i]
        ax.plot(sizes, tr_sc.mean(1), 'o-', label='Treino',    color='#1f4e79', lw=2)
        ax.fill_between(sizes, tr_sc.mean(1)-tr_sc.std(1),
                        tr_sc.mean(1)+tr_sc.std(1), alpha=0.12, color='#1f4e79')
        ax.plot(sizes, val_sc.mean(1), 's--', label='Validação', color='#c0392b', lw=2)
        ax.fill_between(sizes, val_sc.mean(1)-val_sc.std(1),
                        val_sc.mean(1)+val_sc.std(1), alpha=0.12, color='#c0392b')
        gap  = tr_sc.mean(1)[-1] - val_sc.mean(1)[-1]
        diag = ('overfitting'  if gap > 0.15 else
                'underfitting' if val_sc.mean(1)[-1] < 0.3 else 'OK')
        ax.set_title(f"{target.replace('TARGET_', '')}\\n{melhor_nome} (Walk-Forward CV)",
                     fontsize=10, fontweight='bold')
        ax.set_xlabel(f'Tamanho do treino  |  Gap={gap:.2f} → {diag}', fontsize=9)
        ax.set_ylabel('R²')
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        logger.info("Curva %s/%s: gap=%.3f diag=%s", target, melhor_nome, gap, diag)
    except Exception as e:
        logger.warning("Curva de aprendizado falhou %s/%s: %s", target, melhor_nome, e)
        axes[i].text(0.5, 0.5, 'Erro na curva\\n'+str(e)[:60],
                     ha='center', va='center', transform=axes[i].transAxes, fontsize=9)
plt.suptitle('Curvas de Aprendizado — Diagnóstico de Bias/Variância (Walk-Forward CV)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'curvas_aprendizado.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: curvas_aprendizado.png")

2026-05-08 23:50:17 | INFO     | Walk-Forward CV: 5 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


\nGerando curvas de aprendizado...


2026-05-08 23:50:18 | INFO     | Curva TARGET_DRE_3.01/GradientBoosting: gap=-0.002 diag=OK
2026-05-08 23:50:18 | INFO     | Walk-Forward CV: 5 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:50:19 | INFO     | Curva TARGET_DRE_3.11/GradientBoosting: gap=0.601 diag=overfitting
2026-05-08 23:50:19 | INFO     | Walk-Forward CV: 5 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:50:20 | INFO     | Curva TARGET_EBITDA/GradientBoosting: gap=0.082 diag=OK
2026-05-08 23:50:20 | INFO     | Walk-Forward CV: 5 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
2026-05-08 23:50:21 | INFO     | Curva TARGET_BPA_1/GradientBoosting: gap=-0.026 diag=OK
2026-05-08 23:50:21 | INFO     | Walk-Forward CV: 5 folds | anos validação: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
20

  ✅ Salvo: curvas_aprendizado.png


## Etapa 8 — Análise de Resíduos

CORREÇÃO 5: adicionada estratificação dos resíduos por setor.
Três painéis por target:
  1) Predito × Observado (colorido por setor)
  2) Resíduos × Predito  (colorido por setor + detector de funil)
  3) SMAPE por setor     (identifica setores com maior erro relativo)

In [11]:
print("\\nGerando análise de resíduos...")
cols_setor_disp = [c for c in teste.columns if c.startswith('setor_')]
n_t = len(TARGETS)
fig, axes = plt.subplots(n_t, 3, figsize=(21, 5*n_t))
if n_t == 1: axes = axes.reshape(1, -1)
CORES_SETOR = ['#1f4e79','#c0392b','#27ae60','#f39c12','#8e44ad',
               '#2980b9','#e74c3c','#16a085','#d35400','#2c3e50']
for i, target in enumerate(TARGETS):
    transformacao = get_target_transform(target)
    df_te = teste[FEATURES + [target] + cols_setor_disp].copy()
    df_te = df_te[df_te[target].notna()].reset_index(drop=True)
    X_te     = df_te[FEATURES].values
    y_te     = df_te[target].values
    melhor_nome = melhores[target]
    mod         = resultados[target][melhor_nome][0]
    y_pred      = target_inverse_transform(mod.predict(X_te), transformacao)
    residuos    = y_te - y_pred

    setor_labels    = None
    setores_unicos  = []
    if cols_setor_disp:
        setor_labels   = df_te[cols_setor_disp].idxmax(axis=1).str.replace('setor_', '')
        setores_unicos = sorted(setor_labels.unique())

    smape_val = metricas_teste[target][melhor_nome]['SMAPE_teste']
    r2_val    = metricas_teste[target][melhor_nome]['R2_teste']

    # ── Gráfico 1: Predito × Observado ───────────────────────────────────
    ax1 = axes[i, 0]
    if setor_labels is not None:
        for k, setor in enumerate(setores_unicos):
            mask_s = (setor_labels == setor).values
            ax1.scatter(y_pred[mask_s], y_te[mask_s], alpha=0.55, s=22,
                        color=CORES_SETOR[k % len(CORES_SETOR)],
                        label=setor, edgecolors='none')
        ax1.legend(fontsize=7, loc='upper left')
    else:
        ax1.scatter(y_pred, y_te, alpha=0.45, s=18, color='#1f4e79', edgecolors='none')

    lim_max = max(np.nanmax(np.abs(y_te)), np.nanmax(np.abs(y_pred))) * 1.05
    lim_min = min(np.nanmin(y_te), np.nanmin(y_pred)) * 1.05
    ax1.plot([lim_min, lim_max], [lim_min, lim_max], 'r--', lw=1.5)
    ax1.set_xlabel('Predito (R$ mil)')
    ax1.set_ylabel('Observado (R$ mil)')
    ax1.set_title(f"{target.replace('TARGET_', '')} — {melhor_nome}\\nPredito × Observado",
                  fontsize=10, fontweight='bold')
    ax1.text(0.05, 0.92, f'R²={r2_val:.3f}  SMAPE={smape_val:.1%}',
             transform=ax1.transAxes, fontsize=8,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

    # ── Gráfico 2: Resíduos × Predito ────────────────────────────────────
    ax2 = axes[i, 1]
    if setor_labels is not None:
        for k, setor in enumerate(setores_unicos):
            mask_s = (setor_labels == setor).values
            ax2.scatter(y_pred[mask_s], residuos[mask_s], alpha=0.55, s=22,
                        color=CORES_SETOR[k % len(CORES_SETOR)],
                        label=setor, edgecolors='none')
    else:
        ax2.scatter(y_pred, residuos, alpha=0.45, s=18, color='#744210', edgecolors='none')

    ax2.axhline(0,                  color='r',    lw=1.5, ls='--')
    ax2.axhline( np.std(residuos),  color='gray', lw=1,   ls=':', alpha=0.7)
    ax2.axhline(-np.std(residuos),  color='gray', lw=1,   ls=':', alpha=0.7)
    ax2.set_xlabel('Predito (R$ mil)')
    ax2.set_ylabel('Resíduo (R$ mil)')
    ax2.set_title(f'Resíduos × Predito\\nskew={pd.Series(residuos).skew():.2f}'
                  f'  σ={np.std(residuos):,.0f}', fontsize=10, fontweight='bold')

    # Detector de heteroscedasticidade (formato de funil)
    corr_funil = np.corrcoef(np.abs(y_pred), np.abs(residuos))[0, 1]
    if abs(corr_funil) > 0.4:
        ax2.text(0.05, 0.95, f'⚠️ Possível funil\\ncorr={corr_funil:.2f}',
                 transform=ax2.transAxes, fontsize=8, color='red', va='top',
                 bbox=dict(boxstyle='round', facecolor='#fff3cd', alpha=0.8))

    # ── Gráfico 3: SMAPE por setor (CORREÇÃO 5) ───────────────────────────
    ax3 = axes[i, 2]
    if setor_labels is not None:
        smape_por_setor = {}
        for setor in setores_unicos:
            mask_s = (setor_labels == setor).values
            smape_por_setor[setor] = smape(y_te[mask_s], y_pred[mask_s]) if mask_s.sum() > 0 else np.nan

        setores_ord = sorted(smape_por_setor, key=lambda s: smape_por_setor[s] or 0)
        vals  = [smape_por_setor[s] for s in setores_ord]
        cores = [CORES_SETOR[setores_unicos.index(s) % len(CORES_SETOR)] for s in setores_ord]
        bars  = ax3.barh(setores_ord, vals, color=cores, alpha=0.85)
        ax3.axvline(smape_val, color='red', lw=1.5, ls='--', label=f'Média={smape_val:.1%}')
        ax3.set_xlabel('SMAPE')
        ax3.set_title('SMAPE por Setor', fontsize=10, fontweight='bold')
        ax3.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.0%}'))
        ax3.legend(fontsize=8)
        for bar, val in zip(bars, vals):
            if val is not None and np.isfinite(val):
                ax3.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                         f'{val:.1%}', va='center', fontsize=8)
    else:
        ax3.text(0.5, 0.5, 'Setores não disponíveis',
                 ha='center', va='center', transform=ax3.transAxes, fontsize=10)
plt.suptitle('Análise de Resíduos — Teste 2023–2024 (estratificado por setor)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PASTA_SAIDA / 'analise_residuos.png', dpi=150, bbox_inches='tight')
plt.close()
print("  ✅ Salvo: analise_residuos.png")

\nGerando análise de resíduos...
  ✅ Salvo: analise_residuos.png


## Etapa 9 — Persistência completa

In [12]:
rows_cv, rows_te = [], []
for target, algs in resultados.items():
    b = baselines.get(target, {})
    for alg, (_, m) in algs.items():
        rows_cv.append({
            'Target'        : target,
            'Algoritmo'     : alg,
            'RMSE_CV'       : m['RMSE_CV'],
            'RMSE_CV_std'   : m.get('RMSE_CV_std'),
            'SMAPE_CV'      : m['SMAPE_CV'],
            'SMAPE_CV_std'  : m.get('SMAPE_CV_std'),
            'R2_CV'         : m['R2_CV'],
            'TheilU_CV'     : m.get('TheilU_CV'),
            'DA_CV'         : m.get('DA_CV'),
            'n_folds_wf'    : m.get('n_folds_wf'),
            'transformacao' : m.get('transformacao'),
            'log_transform' : m['log_transform'],
            'best_params'   : str(m['best_params']),
        })
        mt = metricas_teste[target][alg]
        rows_te.append({
            'Target'         : target,
            'Algoritmo'      : alg,
            'RMSE_teste'     : mt['RMSE_teste'],
            'MAE_teste'      : mt['MAE_teste'],
            'SMAPE_teste'    : mt['SMAPE_teste'],
            'R2_teste'       : mt['R2_teste'],
            'TheilU_teste'   : mt.get('TheilU_teste'),
            'DA_teste'       : mt.get('DA_teste'),
            'RMSE_baseline'  : b.get('RMSE_baseline'),
            'Bateu_baseline' : mt['RMSE_teste'] < b.get('RMSE_baseline', np.inf),
            'TheilU_ok'      : (mt.get('TheilU_teste', 1.0) or 1.0) < 1.0,
        })
df_cv = pd.DataFrame(rows_cv)
df_te = pd.DataFrame(rows_te)
df_cv.to_csv(PASTA_SAIDA / 'resultados_cv.csv',    index=False)
df_te.to_csv(PASTA_SAIDA / 'resultados_teste.csv', index=False)
with open(PASTA_SAIDA / 'resultados_cv.pkl',       'wb') as f: pickle.dump(resultados, f)
with open(PASTA_SAIDA / 'metricas_teste.pkl',      'wb') as f: pickle.dump(metricas_teste, f)
with open(PASTA_SAIDA / 'baselines.pkl',           'wb') as f: pickle.dump(baselines, f)
with open(PASTA_SAIDA / 'feature_importances.pkl', 'wb') as f: pickle.dump(feature_importances, f)
with open(PASTA_SAIDA / 'melhores_modelos.pkl',    'wb') as f: pickle.dump(melhores, f)
relatorio = {
    'versao'             : 'V6_WalkForward_SMAPE',
    'ano_corte'          : ANO_CORTE,
    'n_treino'           : int(len(treino)),
    'n_teste'            : int(len(teste)),
    'n_splits_wf'        : N_SPLITS_WF,
    'scoring_cv'         : 'SMAPE (proporcional ao tamanho da empresa)',
    'algoritmos'         : list(ALGORITMOS.keys()),
    'targets'            : TARGETS,
    'log_targets'        : list(LOG_TARGETS),
    'arcsinh_targets'    : list(ARCSINH_TARGETS),
    'target_transforms'  : {t: get_target_transform(t) for t in TARGETS},
    'features'           : FEATURES,
    'n_features_lag'     : len(colunas_lag),
    'anos_covid'         : sorted(ANOS_COVID),
    'melhores'           : melhores,
    'baselines'          : {
        t: {k: float(v) for k, v in b.items()
            if isinstance(v, (int, float, np.floating))}
        for t, b in baselines.items()
    },
    'metodologia': {
        'cv_strategy'           : 'Walk-Forward (expanding window) por ano',
        'scoring_otimizacao'    : 'SMAPE — elimina viés de escala entre empresas',
        'selecao_melhor_modelo' : 'SMAPE_CV (sem consultar conjunto de teste)',
        'flag_covid'            : 'Anos 2020–2021 sinalizados explicitamente',
        'residuos'              : 'Estratificados por setor + detector de funil',
    },
}
with open(PASTA_SAIDA / 'relatorio_modelagem.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, indent=2, ensure_ascii=False, default=str)
# ── Resumo final ──────────────────────────────────────────────────────────
print("\\n" + "═"*72)
print("  RESUMO FINAL — Script 3 V6 (Walk-Forward CV + SMAPE scoring)")
print("═"*72)
print(f"  Treino   : {len(treino):,} obs (≤{ANO_CORTE}) | "
      f"DFP={(treino['ORIGEM']=='DFP').sum()} | ITR={(treino['ORIGEM']=='ITR').sum()}")
print(f"  Teste    : {len(teste):,} obs (≥{ANO_CORTE+1}) | "
      f"DFP={(teste['ORIGEM']=='DFP').sum()}  | ITR={(teste['ORIGEM']=='ITR').sum()}")
print(f"  Modelos  : {len(TARGETS) * len(ALGORITMOS)} ({len(TARGETS)} targets × {len(ALGORITMOS)} algoritmos)")
print(f"  CV       : Walk-Forward, {N_SPLITS_WF} folds, scoring=SMAPE")
print(f"  Features : {len(FEATURES)} ({len(colunas_lag)} temporais: lags/yoy/roll)")
print(f"  COVID    : {sorted(ANOS_COVID)} → flag_covid=1 no dataset")
print(f"  Métricas : RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional")
print()
# CORREÇÃO 4: texto explica que seleção é por SMAPE_CV, avaliação é por métricas de teste
print("  Melhores (seleção por SMAPE_CV | avaliação por métricas de teste):")
print(f"  {'Target':<35} {'Algoritmo':<20} {'SMAPE_CV':>9} "
      f"{'SMAPE_te':>9} {'R²_te':>7} {'TheilU':>7} {'Bateu?':>7}")
print(f"  {'-'*35} {'-'*20} {'-'*9} {'-'*9} {'-'*7} {'-'*7} {'-'*7}")
for t, alg in melhores.items():
    m_cv   = resultados[t][alg][1]
    m_test = metricas_teste[t][alg]
    b      = baselines.get(t, {})
    bateu  = m_test['RMSE_teste'] < b.get('RMSE_baseline', np.inf)
    print(f"  {t:<35} {alg:<20} "
          f"{m_cv['SMAPE_CV']:>9.1%} "
          f"{m_test['SMAPE_teste']:>9.1%} "
          f"{m_test['R2_teste']:>7.3f} "
          f"{m_test['TheilU_teste']:>7.3f} "
          f"{'✅' if bateu else '❌':>7}")
print("═"*72)
print("  ✅ Pronto para Script 4 (Avaliação + Z'')")
print("═"*72)

\n════════════════════════════════════════════════════════════════════════
  RESUMO FINAL — Script 3 V6 (Walk-Forward CV + SMAPE scoring)
════════════════════════════════════════════════════════════════════════
  Treino   : 713 obs (≤2022) | DFP=181 | ITR=532
  Teste    : 168 obs (≥2023) | DFP=24  | ITR=144
  Modelos  : 36 (9 targets × 4 algoritmos)
  CV       : Walk-Forward, 5 folds, scoring=SMAPE
  Features : 16 (15 temporais: lags/yoy/roll)
  COVID    : [2020, 2021] → flag_covid=1 no dataset
  Métricas : RMSE, MAE, SMAPE, R², Theil's U, Acurácia Direcional

  Melhores (seleção por SMAPE_CV | avaliação por métricas de teste):
  Target                              Algoritmo             SMAPE_CV  SMAPE_te   R²_te  TheilU  Bateu?
  ----------------------------------- -------------------- --------- --------- ------- ------- -------
  TARGET_DRE_3.01                     GradientBoosting          8.6%      4.1%   0.978   0.301       ✅
  TARGET_DRE_3.11                     GradientBoosting 